In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

In [37]:
bitcoin_df = pd.read_csv('bitcoin-price.csv')
market_df = pd.read_csv('market-data.csv')
onchain_df = pd.read_csv('on-chain-data.csv')

bitcoin_df['Date'] = pd.to_datetime(bitcoin_df['Datetime']).dt.date
market_df['Date'] = pd.to_datetime(market_df['Dates']).dt.date
onchain_df['Date'] = pd.to_datetime(onchain_df['Datetime']).dt.date

bitcoin_close = bitcoin_df[['Date', 'Close']].copy()
print(f"Bitcoin date range: {bitcoin_close['Date'].min()} to {bitcoin_close['Date'].max()}")
print(f"Market date range: {market_df['Date'].min()} to {market_df['Date'].max()}")
print(f"On-chain date range: {onchain_df['Date'].min()} to {onchain_df['Date'].max()}")

Bitcoin date range: 2009-01-03 to 2025-06-10
Market date range: 2017-01-02 to 2025-06-23
On-chain date range: 2019-03-30 to 2025-06-09


In [38]:
bitcoin_market_df = pd.merge(bitcoin_close, market_df, on='Date', how='inner')
bitcoin_market_df = bitcoin_market_df.dropna()

print(f"Shape: {bitcoin_market_df.shape}")
print(f"Date range: {bitcoin_market_df['Date'].min()} to {bitcoin_market_df['Date'].max()}")
print(f"Columns: {list(bitcoin_market_df.columns)}")
print("\nFirst few rows:")
print(bitcoin_market_df.head())

Shape: (2202, 14)
Date range: 2017-01-02 to 2025-06-10
Columns: ['Date', 'Close', 'Dates', 'SPX Index', 'CCMP Index', 'INDU Index', 'XAU Curncy', 'CL1 Comdty', 'XAG Curncy', 'LMCADY Comdty', 'NG1 Comdty', 'DXY Curncy', 'EURUSD Curncy', 'USDCNY Curncy']

First few rows:
         Date          Close  ... EURUSD Curncy  USDCNY Curncy
0  2025-06-10  108923.014410  ...        1.1425         7.1878
1  2025-06-09  110303.871227  ...        1.1422         7.1794
2  2025-06-06  104406.924809  ...        1.1397         7.1926
3  2025-06-05  101613.260806  ...        1.1445         7.1777
4  2025-06-04  104724.591994  ...        1.1417         7.1780

[5 rows x 14 columns]


In [39]:
bitcoin_onchain_df = pd.merge(bitcoin_close, onchain_df, on='Date', how='inner')
bitcoin_onchain_df = bitcoin_onchain_df.dropna()

print(f"Shape: {bitcoin_onchain_df.shape}")
print(f"Date range: {bitcoin_onchain_df['Date'].min()} to {bitcoin_onchain_df['Date'].max()}")
print(f"Columns: {list(bitcoin_onchain_df.columns)}")
print("\nFirst few rows:")
print(bitcoin_onchain_df.head())

Shape: (2264, 15)
Date range: 2019-03-30 to 2025-06-09
Columns: ['Date', 'Close', 'Datetime', 'Active Addresses', 'Exchange Reserve', 'Exchange Netflow', 'Fee-Reward Ratio', 'Funding Rates', 'Hashrate', 'Mean Coin Age', 'Miner Reserve', 'MVRV Ratio', 'Open Interest', 'Puell Multiple', 'SOPR']

First few rows:
         Date          Close  ... Puell Multiple      SOPR
0  2025-06-09  110303.871227  ...       1.243001  1.013669
1  2025-06-08  105782.295477  ...       1.337595  1.011710
2  2025-06-07  105645.525274  ...       1.245541  1.013583
3  2025-06-06  104406.924809  ...       1.152875  1.006889
4  2025-06-05  101613.260806  ...       1.156797  1.002772

[5 rows x 15 columns]
